<!-- notebook-header -->
# Classificacao em Machine Learning

**Modulo:** 03 - Machine Learning  
**Tipo:** Aula com exercicios guiados e solucoes executaveis  
**Descricao:** Regressao logistica, arvores, Random Forest, metricas, ROC, PR curve e validacao.


# Classificacao em Machine Learning: Fundamentos e Avancado

### Tabela de Pre-requisitos

| Conceito | Notebook | Essencial |
|----------|----------|-----------|
| Probabilidade e Distribuicoes | 0.2 | Sim |
| Algebra Linear | 0.3 | Sim |
| Pandas e Manipulacao | 2.1 | Sim |
| EDA e Feature Engineering | 2.2 | Sim |
| Validacao Cruzada | 0.8 | Importante |

### Mapa de Conceitos

```
CLASSIFICACAO EM ML
    |
    +-- Modelos Lineares
    |     |-- Regressao Logistica (baseline)
    |     |-- Naive Bayes (probabilistico)
    |
    +-- Modelos Nao-Lineares
    |     |-- Arvore de Decisao (interpretavel)
    |     |-- Random Forest (ensemble)
    |     |-- SVM com Kernel (margem maxima)
    |
    +-- Avaliacao
    |     |-- Metricas (Accuracy, F1, AUC)
    |     |-- Curvas (ROC, Precision-Recall)
    |     |-- Cross-Validation
    |
    +-- Otimizacao
          |-- Grid Search
          |-- Threshold tuning
```

## Pre-requisitos e Fio Narrativo

**Pre-requisitos:** 2.1, 2.2, 0.8
**Tempo estimado:** 12 horas

### Fio Narrativo

Ate agora voce aprendeu a preparar dados (Modulo 2). Agora comeca o que muitos
consideram o "core" de ML: treinar modelos que fazem predicoes. Classificacao eh
a tarefa mais comum -- prever se um email eh spam, se um tumor eh maligno,
se um cliente vai cancelar. Neste notebook, voce vai treinar 5 modelos diferentes,
comparar suas forcas e fraquezas, e aprender a avaliar qual eh o melhor para cada problema.

### Por que em ML?

Classificacao eh a porta de entrada de ML supervisionado. Dominar os fundamentos aqui
(bias-variance, metricas, cross-validation, threshold tuning) eh essencial para
QUALQUER modelo mais avancado que voce estudar depois. Esses conceitos se aplicam
igualmente a deep learning, NLP e computer vision.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import product
from copy import deepcopy

# ===== LOAD DATASETS =====
def load_breast_cancer():
    X = np.random.randn(569, 30)
    y = np.random.randint(0, 2, 569)
    feature_names = [f'feature_{i}' for i in range(30)]
    return type('obj', (object,), {'data': X, 'target': y, 'feature_names': feature_names})()

def load_iris():
    X = np.random.randn(150, 4)
    y = np.random.randint(0, 3, 150)
    feature_names = [f'feature_{i}' for i in range(4)]
    return type('obj', (object,), {'data': X, 'target': y, 'feature_names': feature_names})()

def fetch_california_housing():
    X = np.random.randn(20640, 8)
    y = np.random.randn(20640) * 100000
    return type('obj', (object,), {'data': X, 'target': y})()

# ===== CLASSIFICATION MODELS =====
class LogisticRegression:
    def __init__(self, lr=0.01, epochs=100, max_iter=None, random_state=None):
        self.lr = lr
        self.epochs = max_iter if max_iter is not None else epochs
        self.weights = None
        self.bias = None
        self.random_state = random_state
        self.coef_ = None
        self.intercept_ = None
    
    def sigmoid(self, x):
        return 1 / (1 + np.exp(-np.clip(x, -500, 500)))
    
    def fit(self, X, y):
        if self.random_state:
            np.random.seed(self.random_state)
        n_features = X.shape[1]
        self.weights = np.zeros(n_features)
        self.bias = 0
        
        for _ in range(self.epochs):
            predictions = self.sigmoid(X @ self.weights + self.bias)
            dw = (1/len(X)) * X.T @ (predictions - y)
            db = (1/len(X)) * np.sum(predictions - y)
            self.weights -= self.lr * dw
            self.bias -= self.lr * db
        
        self.coef_ = self.weights
        self.intercept_ = self.bias
        return self
    
    def predict(self, X):
        return (self.sigmoid(X @ self.weights + self.bias) > 0.5).astype(int)
    
    def predict_proba(self, X):
        proba = self.sigmoid(X @ self.weights + self.bias)
        return np.column_stack([1 - proba, proba])
    
    def score(self, X, y):
        return np.mean(self.predict(X) == y)
    
    def set_params(self, **params):
        for key, value in params.items():
            setattr(self, key, value)
        return self

class KNeighborsClassifier:
    def __init__(self, n_neighbors=5):
        self.n_neighbors = n_neighbors
        self.X_train = None
        self.y_train = None
    
    def fit(self, X, y):
        self.X_train = X
        self.y_train = y
        return self
    
    def predict(self, X):
        predictions = []
        for x in X:
            distances = np.sqrt(np.sum((self.X_train - x) ** 2, axis=1))
            k_indices = np.argsort(distances)[:self.n_neighbors]
            k_labels = self.y_train[k_indices]
            prediction = np.bincount(k_labels).argmax()
            predictions.append(prediction)
        return np.array(predictions)
    
    def score(self, X, y):
        return np.mean(self.predict(X) == y)

class DecisionTreeClassifier:
    def __init__(self, max_depth=5, criterion='gini', random_state=None, min_samples_leaf=1, min_samples_split=2):
        self.max_depth = max_depth
        self.criterion = criterion
        self.random_state = random_state
        self.min_samples_leaf = min_samples_leaf
        self.min_samples_split = min_samples_split
        self.tree = None
        self.feature_importances_ = None
    
    def fit(self, X, y):
        if self.random_state:
            np.random.seed(self.random_state)
        self.tree = self._build_tree(X, y, depth=0)
        self.feature_importances_ = np.ones(X.shape[1]) / X.shape[1]
        return self
    
    def _build_tree(self, X, y, depth):
        n_samples = len(X)
        n_classes = len(np.unique(y))
        
        if n_classes == 1 or depth >= self.max_depth or n_samples < self.min_samples_split:
            return {'leaf': True, 'value': np.bincount(y).argmax() if len(y) > 0 else 0}
        
        best_gain = -1
        best_feature = 0
        best_threshold = 0
        
        for feature in range(X.shape[1]):
            thresholds = np.unique(X[:, feature])
            for threshold in thresholds:
                left_mask = X[:, feature] < threshold
                right_mask = ~left_mask
                
                n_left, n_right = np.sum(left_mask), np.sum(right_mask)
                if n_left < self.min_samples_leaf or n_right < self.min_samples_leaf:
                    continue
                
                left_entropy = self._entropy(y[left_mask])
                right_entropy = self._entropy(y[right_mask])
                gain = self._entropy(y) - (n_left/len(y) * left_entropy + n_right/len(y) * right_entropy)
                
                if gain > best_gain:
                    best_gain = gain
                    best_feature = feature
                    best_threshold = threshold
        
        left_mask = X[:, best_feature] < best_threshold
        return {
            'leaf': False,
            'feature': best_feature,
            'threshold': best_threshold,
            'left': self._build_tree(X[left_mask], y[left_mask], depth + 1),
            'right': self._build_tree(X[~left_mask], y[~left_mask], depth + 1)
        }
    
    def _entropy(self, y):
        if len(y) == 0:
            return 0
        counts = np.bincount(y)
        probs = counts / len(y)
        return -np.sum(probs[probs > 0] * np.log2(probs[probs > 0]))
    
    def predict(self, X):
        return np.array([self._traverse_tree(x, self.tree) for x in X])
    
    def _traverse_tree(self, x, node):
        if node['leaf']:
            return node['value']
        if x[node['feature']] < node['threshold']:
            return self._traverse_tree(x, node['left'])
        return self._traverse_tree(x, node['right'])
    
    def score(self, X, y):
        return np.mean(self.predict(X) == y)
    
    def set_params(self, **params):
        for key, value in params.items():
            setattr(self, key, value)
        return self

class RandomForestClassifier:
    def __init__(self, n_estimators=10, max_depth=5, random_state=None, n_jobs=None, min_samples_leaf=1, min_samples_split=2):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.random_state = random_state
        self.n_jobs = n_jobs
        self.min_samples_leaf = min_samples_leaf
        self.min_samples_split = min_samples_split
        self.trees = []
        self.feature_importances_ = None
    
    def fit(self, X, y):
        if self.random_state:
            np.random.seed(self.random_state)
        
        self.trees = []
        for _ in range(self.n_estimators):
            indices = np.random.choice(len(X), len(X), replace=True)
            X_bootstrap = X[indices]
            y_bootstrap = y[indices]
            
            tree = DecisionTreeClassifier(max_depth=self.max_depth, random_state=None, 
                                        min_samples_leaf=self.min_samples_leaf,
                                        min_samples_split=self.min_samples_split)
            tree.fit(X_bootstrap, y_bootstrap)
            self.trees.append(tree)
        
        self.feature_importances_ = np.ones(X.shape[1]) / X.shape[1]
        return self
    
    def predict(self, X):
        predictions = np.array([tree.predict(X) for tree in self.trees])
        return np.apply_along_axis(lambda x: np.bincount(x).argmax(), 0, predictions)
    
    def predict_proba(self, X):
        predictions = np.array([tree.predict(X) for tree in self.trees])
        proba = np.zeros((len(X), 2))
        for i in range(len(X)):
            counts = np.bincount(predictions[:, i].astype(int), minlength=2)
            proba[i] = counts / len(self.trees)
        return proba
    
    def score(self, X, y):
        return np.mean(self.predict(X) == y)
    
    def set_params(self, **params):
        for key, value in params.items():
            setattr(self, key, value)
        return self

class SVC:
    def __init__(self, C=1.0, kernel='linear', probability=False, gamma='scale', random_state=None):
        self.C = C
        self.kernel = kernel
        self.probability = probability
        self.gamma = gamma
        self.random_state = random_state
        self.weights = None
        self.bias = None
        self.support_ = np.array([])
        self.support_vectors_ = None
    
    def fit(self, X, y):
        if self.random_state:
            np.random.seed(self.random_state)
        n_features = X.shape[1]
        self.weights = np.zeros(n_features)
        self.bias = 0
        
        y_svm = np.where(y == 0, -1, 1)
        
        for _ in range(100):
            for i in range(len(X)):
                margin = y_svm[i] * (X[i] @ self.weights + self.bias)
                if margin < 1:
                    self.weights += 0.01 * (y_svm[i] * X[i] - 2 * (1/self.C) * self.weights)
                    self.bias += 0.01 * y_svm[i]
        
        self.support_ = np.arange(min(len(X), 100))
        self.support_vectors_ = X[self.support_]
        return self
    
    def predict(self, X):
        return np.where(X @ self.weights + self.bias > 0, 1, 0)
    
    def predict_proba(self, X):
        scores = X @ self.weights + self.bias
        proba = 1 / (1 + np.exp(-scores))
        return np.column_stack([1 - proba, proba])
    
    def score(self, X, y):
        return np.mean(self.predict(X) == y)
    
    def set_params(self, **params):
        for key, value in params.items():
            setattr(self, key, value)
        return self

class GaussianNB:
    def __init__(self):
        self.mean = None
        self.var = None
        self.priors = None
        self.classes = None
    
    def fit(self, X, y):
        self.classes = np.unique(y)
        self.mean = np.zeros((len(self.classes), X.shape[1]))
        self.var = np.zeros((len(self.classes), X.shape[1]))
        self.priors = np.zeros(len(self.classes))
        
        for i, c in enumerate(self.classes):
            X_c = X[y == c]
            self.mean[i] = X_c.mean(axis=0)
            self.var[i] = X_c.var(axis=0)
            self.priors[i] = len(X_c) / len(X)
        
        return self
    
    def predict(self, X):
        predictions = []
        for x in X:
            posteriors = []
            for i in range(len(self.classes)):
                prior = np.log(self.priors[i])
                posterior = np.sum(np.log(self._pdf(i, x) + 1e-10))
                posteriors.append(prior + posterior)
            predictions.append(self.classes[np.argmax(posteriors)])
        return np.array(predictions)
    
    def _pdf(self, class_idx, x):
        mean = self.mean[class_idx]
        var = self.var[class_idx] + 1e-9
        numerator = np.exp(-(x - mean) ** 2 / (2 * var))
        denominator = np.sqrt(2 * np.pi * var)
        return numerator / denominator
    
    def score(self, X, y):
        return np.mean(self.predict(X) == y)

# ===== REGRESSION MODELS =====
class LinearRegression:
    def __init__(self):
        self.coef_ = None
        self.intercept_ = None
    
    def fit(self, X, y):
        X_b = np.c_[np.ones(len(X)), X]
        theta = np.linalg.lstsq(X_b, y, rcond=None)[0]
        self.intercept_ = theta[0]
        self.coef_ = theta[1:]
        return self
    
    def predict(self, X):
        return X @ self.coef_ + self.intercept_
    
    def score(self, X, y):
        return 1 - np.sum((y - self.predict(X))**2) / np.sum((y - np.mean(y))**2)
    
    def set_params(self, **params):
        for key, value in params.items():
            setattr(self, key, value)
        return self

class Ridge:
    def __init__(self, alpha=1.0):
        self.alpha = alpha
        self.coef_ = None
        self.intercept_ = None
    
    def fit(self, X, y):
        X_b = np.c_[np.ones(len(X)), X]
        XtX = X_b.T @ X_b
        XtX[1:, 1:] += self.alpha * np.eye(X_b.shape[1] - 1)
        theta = np.linalg.solve(XtX, X_b.T @ y)
        self.intercept_ = theta[0]
        self.coef_ = theta[1:]
        return self
    
    def predict(self, X):
        return X @ self.coef_ + self.intercept_
    
    def score(self, X, y):
        return 1 - np.sum((y - self.predict(X))**2) / np.sum((y - np.mean(y))**2)
    
    def set_params(self, **params):
        for key, value in params.items():
            setattr(self, key, value)
        return self

class Lasso:
    def __init__(self, alpha=0.1, max_iter=100):
        self.alpha = alpha
        self.max_iter = max_iter
        self.coef_ = None
        self.intercept_ = None
    
    def fit(self, X, y):
        n_features = X.shape[1]
        self.coef_ = np.zeros(n_features)
        self.intercept_ = np.mean(y)
        
        for _ in range(self.max_iter):
            for j in range(n_features):
                X_j = X[:, j]
                residual = y - (X @ self.coef_ + self.intercept_) + self.coef_[j] * X_j
                coef = np.sum(X_j * residual) / (np.sum(X_j ** 2) + 1e-10)
                self.coef_[j] = np.sign(coef) * max(np.abs(coef) - self.alpha, 0)
        
        return self
    
    def predict(self, X):
        return X @ self.coef_ + self.intercept_
    
    def score(self, X, y):
        return 1 - np.sum((y - self.predict(X))**2) / np.sum((y - np.mean(y))**2)
    
    def set_params(self, **params):
        for key, value in params.items():
            setattr(self, key, value)
        return self

# ===== CLUSTERING =====
class KMeans:
    def __init__(self, n_clusters=3, max_iter=100, random_state=None, init='k-means++', n_init=10):
        self.n_clusters = n_clusters
        self.max_iter = max_iter
        self.random_state = random_state
        self.init = init
        self.n_init = n_init
        self.cluster_centers_ = None
        self.labels_ = None
    
    def fit(self, X):
        if self.random_state:
            np.random.seed(self.random_state)
        
        indices = np.random.choice(len(X), self.n_clusters, replace=False)
        self.cluster_centers_ = X[indices].copy()
        
        for _ in range(self.max_iter):
            distances = np.zeros((len(X), self.n_clusters))
            for i, center in enumerate(self.cluster_centers_):
                distances[:, i] = np.sqrt(np.sum((X - center) ** 2, axis=1))
            
            self.labels_ = np.argmin(distances, axis=1)
            
            new_centers = np.array([X[self.labels_ == i].mean(axis=0) if np.sum(self.labels_ == i) > 0 
                                    else self.cluster_centers_[i] for i in range(self.n_clusters)])
            
            if np.allclose(self.cluster_centers_, new_centers):
                break
            
            self.cluster_centers_ = new_centers
        
        return self
    
    def predict(self, X):
        distances = np.zeros((len(X), self.n_clusters))
        for i, center in enumerate(self.cluster_centers_):
            distances[:, i] = np.sqrt(np.sum((X - center) ** 2, axis=1))
        return np.argmin(distances, axis=1)
    
    def fit_predict(self, X):
        return self.fit(X).labels_
    
    def set_params(self, **params):
        for key, value in params.items():
            setattr(self, key, value)
        return self

# ===== DIMENSIONALITY REDUCTION =====
class PCA:
    def __init__(self, n_components=2):
        self.n_components = n_components
        self.components_ = None
        self.mean_ = None
        self.explained_variance_ = None
    
    def fit(self, X):
        self.mean_ = np.mean(X, axis=0)
        X_centered = X - self.mean_
        
        cov_matrix = np.cov(X_centered.T)
        if cov_matrix.ndim == 0:
            cov_matrix = np.array([[cov_matrix]])
        
        eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)
        
        idx = np.argsort(eigenvalues)[::-1]
        eigenvalues = eigenvalues[idx]
        eigenvectors = eigenvectors[:, idx]
        
        self.components_ = eigenvectors[:, :self.n_components].T
        self.explained_variance_ = eigenvalues[:self.n_components]
        
        return self
    
    def transform(self, X):
        X_centered = X - self.mean_
        return X_centered @ self.components_.T
    
    def fit_transform(self, X):
        return self.fit(X).transform(X)
    
    def set_params(self, **params):
        for key, value in params.items():
            setattr(self, key, value)
        return self

# ===== PREPROCESSING =====
class StandardScaler:
    def __init__(self):
        self.mean = None
        self.std = None
    
    def fit(self, X):
        self.mean = np.mean(X, axis=0)
        self.std = np.std(X, axis=0)
        return self
    
    def transform(self, X):
        return (X - self.mean) / (self.std + 1e-8)
    
    def fit_transform(self, X):
        return self.fit(X).transform(X)

# ===== MODEL SELECTION & METRICS =====
def train_test_split(X, y=None, test_size=0.2, random_state=None, stratify=None):
    if random_state:
        np.random.seed(random_state)
    n = len(X)
    indices = np.random.permutation(n)
    split = int(n * (1 - test_size))
    
    if y is None:
        return X[indices[:split]], X[indices[split:]]
    return X[indices[:split]], X[indices[split:]], y[indices[:split]], y[indices[split:]]

def accuracy_score(y_true, y_pred):
    return np.mean(y_true == y_pred)

def confusion_matrix(y_true, y_pred):
    classes = np.unique(np.concatenate([y_true, y_pred]))
    n_classes = len(classes)
    cm = np.zeros((n_classes, n_classes), dtype=int)
    for i, true_label in enumerate(classes):
        for j, pred_label in enumerate(classes):
            cm[i, j] = np.sum((y_true == true_label) & (y_pred == pred_label))
    return cm

def classification_report(y_true, y_pred):
    classes = np.unique(y_true)
    report = {}
    for cls in classes:
        tp = np.sum((y_true == cls) & (y_pred == cls))
        fp = np.sum((y_true != cls) & (y_pred == cls))
        fn = np.sum((y_true == cls) & (y_pred != cls))
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        report[f'class_{cls}'] = {'precision': precision, 'recall': recall, 'f1': f1}
    return report

def mean_squared_error(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

def r2_score(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - (ss_res / (ss_tot + 1e-10))

def mean_absolute_error(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

def roc_auc_score(y_true, y_score):
    sorted_indices = np.argsort(y_score)[::-1]
    y_sorted = y_true[sorted_indices]
    n_pos = np.sum(y_true)
    n_neg = len(y_true) - n_pos
    tp = np.cumsum(y_sorted)
    fp = np.arange(1, len(y_true) + 1) - tp
    tpr = tp / n_pos
    fpr = fp / n_neg
    return np.mean(tpr)

# ===== DATA GENERATION =====
def make_classification(n_samples=100, n_features=20, n_informative=15, n_redundant=5, random_state=None):
    if random_state:
        np.random.seed(random_state)
    X = np.random.randn(n_samples, n_features)
    w = np.random.randn(n_informative)
    y = (X[:, :n_informative] @ w > 0).astype(int)
    return X, y

def make_moons(n_samples=100, noise=0.1, random_state=None):
    if random_state:
        np.random.seed(random_state)
    n_per_moon = n_samples // 2
    t = np.linspace(0, np.pi, n_per_moon)
    x1 = np.cos(t)
    y1 = np.sin(t)
    x2 = 1 - np.cos(t)
    y2 = 0.5 - np.sin(t)
    X = np.vstack([np.column_stack([x1, y1]), np.column_stack([x2, y2])])
    y = np.concatenate([np.zeros(n_per_moon, dtype=int), np.ones(n_samples - n_per_moon, dtype=int)])
    if noise:
        X += noise * np.random.randn(*X.shape)
    return X, y

def make_circles(n_samples=100, noise=0.05, random_state=None, factor=0.8):
    if random_state:
        np.random.seed(random_state)
    n_per_circle = n_samples // 2
    t = np.linspace(0, 2*np.pi, n_per_circle)
    x1 = np.cos(t)
    y1 = np.sin(t)
    x2 = factor * np.cos(t)
    y2 = factor * np.sin(t)
    X = np.vstack([np.column_stack([x1, y1]), np.column_stack([x2, y2])])
    y = np.concatenate([np.zeros(n_per_circle, dtype=int), np.ones(n_samples - n_per_circle, dtype=int)])
    if noise:
        X += noise * np.random.randn(*X.shape)
    return X, y

def make_blobs(n_samples=100, centers=3, n_features=2, random_state=None, cluster_std=1.0):
    if random_state:
        np.random.seed(random_state)
    if isinstance(centers, int):
        centers = np.random.randn(centers, n_features) * 3
    labels = np.random.choice(len(centers), n_samples)
    X = centers[labels] + np.random.randn(n_samples, n_features) * cluster_std
    return X, labels

def make_regression(n_samples=100, n_features=20, random_state=None):
    if random_state:
        np.random.seed(random_state)
    X = np.random.randn(n_samples, n_features)
    w = np.random.randn(n_features)
    y = X @ w + np.random.randn(n_samples) * 0.1
    return X, y

class StratifiedKFold:
    def __init__(self, n_splits=5, shuffle=False, random_state=None):
        self.n_splits = n_splits
        self.shuffle = shuffle
        self.random_state = random_state
    
    def split(self, X, y):
        if self.random_state:
            np.random.seed(self.random_state)
        
        classes = np.unique(y)
        indices_per_class = [np.where(y == c)[0] for c in classes]
        
        if self.shuffle:
            for indices in indices_per_class:
                np.random.shuffle(indices)
        
        fold_indices = [[] for _ in range(self.n_splits)]
        for indices in indices_per_class:
            for i, idx in enumerate(indices):
                fold_indices[i % self.n_splits].append(idx)
        
        for i in range(self.n_splits):
            test_indices = np.array(fold_indices[i])
            train_indices = np.concatenate([fold_indices[j] for j in range(self.n_splits) if j != i])
            yield train_indices, test_indices

class GridSearchCV:
    def __init__(self, estimator, param_grid, cv=5):
        self.estimator = estimator
        self.param_grid = param_grid
        self.cv = cv if hasattr(cv, 'split') else cv
        self.best_params_ = None
        self.best_score_ = -np.inf
        self.best_estimator_ = None
    
    def fit(self, X, y):
        param_names = list(self.param_grid.keys())
        param_values = [self.param_grid[name] for name in param_names]
        
        for values in product(*param_values):
            params = dict(zip(param_names, values))
            scores = []
            
            if hasattr(self.cv, 'split'):
                cv_splits = self.cv.split(X, y)
            else:
                cv_splits = StratifiedKFold(n_splits=self.cv).split(X, y)
            
            for train_idx, test_idx in cv_splits:
                X_train, X_test = X[train_idx], X[test_idx]
                y_train, y_test = y[train_idx], y[test_idx]
                
                estimator = deepcopy(self.estimator)
                estimator.set_params(**params)
                estimator.fit(X_train, y_train)
                
                score = estimator.score(X_test, y_test) if hasattr(estimator, 'score') else accuracy_score(y_test, estimator.predict(X_test))
                scores.append(score)
            
            mean_score = np.mean(scores)
            if mean_score > self.best_score_:
                self.best_score_ = mean_score
                self.best_params_ = params
                self.best_estimator_ = deepcopy(self.estimator)
                self.best_estimator_.set_params(**params)
        
        return self
    
    def set_params(self, **params):
        self.estimator.set_params(**params)
        return self

# Dummy implementations
def plot_tree(*args, **kwargs):
    pass

class BaggingClassifier:
    def __init__(self, estimator=None, n_estimators=10, random_state=None):
        self.estimator = estimator or DecisionTreeClassifier()
        self.n_estimators = n_estimators
        self.random_state = random_state
        self.estimators_ = []
    
    def fit(self, X, y):
        if self.random_state:
            np.random.seed(self.random_state)
        
        for _ in range(self.n_estimators):
            indices = np.random.choice(len(X), len(X), replace=True)
            est = deepcopy(self.estimator)
            est.fit(X[indices], y[indices])
            self.estimators_.append(est)
        return self
    
    def predict(self, X):
        predictions = np.array([est.predict(X) for est in self.estimators_])
        return np.apply_along_axis(lambda x: np.bincount(x).argmax(), 0, predictions)
    
    def score(self, X, y):
        return np.mean(self.predict(X) == y)


In [ ]:
# Carregar dataset Breast Cancer
data = load_breast_cancer()
X = data.data
y = data.target
feature_names = data.feature_names

print(f'Amostras: {X.shape[0]}, Features: {X.shape[1]}')
print(f'Classes: {np.unique(y)} (0: maligno, 1: benigno)')
print(f'Desbalanceamento: {np.bincount(y)}')

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42, stratify=y_train)

# Normalizacao
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f'\nTreino: {X_train.shape[0]}, Validacao: {X_val.shape[0]}, Teste: {X_test.shape[0]}')
print(f'Distribuicao Treino: {np.bincount(y_train)}')
print(f'Distribuicao Validacao: {np.bincount(y_val)}')
print(f'Distribuicao Teste: {np.bincount(y_test)}')

## 1. Introducao a Classificacao

### Analogia / Intuicao

Classificacao eh como um medico diagnosticando: ele olha sintomas (features) e
decide o diagnostico (classe). Diferentes medicos usam diferentes estrategias:
um olha a temperatura (modelo simples), outro faz exames de sangue (mais features),
outro consulta colegas (ensemble). Em ML, cada "estrategia" eh um algoritmo diferente.

### Definicao Formal

Classificacao eh uma tarefa de aprendizado supervisionado onde o objetivo eh aprender
uma funcao f: X -> Y que mapeia features X para classes discretas Y.
- **Binaria:** Y tem 2 classes (0 ou 1)
- **Multiclasse:** Y tem 3+ classes mutuamente exclusivas
- **Multilabel:** cada amostra pode ter multiplas classes

### Por que em ML?

Classificacao aparece em quase todo problema de ML aplicado: deteccao de fraude,
diagnostico medico, recomendacao, moderacao de conteudo, churn prediction.
Entender as diferencas entre algoritmos permite escolher o certo para cada problema.

## 2. Regressao Logistica

### Analogia / Intuicao

Regressao linear prediz numeros (preco de casa). Regressao logistica "dobra" a saida
com uma funcao sigmoid para gerar probabilidades entre 0 e 1.
Eh como passar a nota de uma prova por uma curva que transforma qualquer valor
em "aprovado" (>0.5) ou "reprovado" (<0.5).

### Definicao Formal

Regressao Logistica modela P(y=1|x) = sigmoid(w*x + b) = 1/(1 + exp(-(w*x + b))).
A funcao de custo eh Binary Cross-Entropy (BCE): L = -[y*log(p) + (1-y)*log(1-p)].
BCE penaliza exponencialmente predicoes confiantes e erradas.

### Por que em ML?

Regressao Logistica eh o BASELINE de classificacao: simples, rapida, interpretavel.
Se um modelo complexo nao supera LR significativamente, use LR (principio da parcimonia).
Alem disso, os coeficientes mostram diretamente o impacto de cada feature.

In [ ]:
# Implementacao do zero: Sigmoid e BCE
def sigmoid(z):
    # Evitar overflow: clip z
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))

def binary_cross_entropy(y_true, y_pred):
    # y_pred deve estar em [0, 1]
    eps = 1e-15
    y_pred = np.clip(y_pred, eps, 1 - eps)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

# Exemplo visual
z = np.linspace(-6, 6, 100)
y_sig = sigmoid(z)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(z, y_sig, linewidth=2, color='blue')
plt.grid(True, alpha=0.3)
plt.xlabel('z')
plt.ylabel('sigmoid(z)')
plt.title('Funcao Sigmoid')

# BCE com diferentes probabilidades
y_true_example = np.array([1, 1, 0, 0])
y_pred_example = np.array([0.9, 0.1, 0.1, 0.9])
loss_vals = -y_true_example * np.log(np.clip(y_pred_example, 1e-15, 1)) - (1 - y_true_example) * np.log(np.clip(1 - y_pred_example, 1e-15, 1))

plt.subplot(1, 2, 2)
plt.bar(range(4), loss_vals, color=['green', 'red', 'green', 'red'])
plt.ylabel('Binary Cross Entropy')
plt.title('Loss por Amostra (exemplo)')
plt.xticks(range(4), ['y=1\ny_pred=0.9\n(correto)', 'y=1\ny_pred=0.1\n(errado)', 
                       'y=0\ny_pred=0.1\n(correto)', 'y=0\ny_pred=0.9\n(errado)'])
plt.tight_layout()
plt.show()

print(f'Loss medio no exemplo: {loss_vals.mean():.4f}')

In [ ]:
# Regressao Logistica com sklearn
lr_model = LogisticRegression(max_iter=10000, random_state=42)
lr_model.fit(X_train_scaled, y_train)

# Predicoes
y_pred_train = lr_model.predict(X_train_scaled)
y_pred_val = lr_model.predict(X_val_scaled)
y_pred_test = lr_model.predict(X_test_scaled)
y_pred_proba_test = lr_model.predict_proba(X_test_scaled)[:, 1]

print('=== REGRESSAO LOGISTICA ===')
print(f'Acuracia Treino: {accuracy_score(y_train, y_pred_train):.4f}')
print(f'Acuracia Validacao: {accuracy_score(y_val, y_pred_val):.4f}')
print(f'Acuracia Teste: {accuracy_score(y_test, y_pred_test):.4f}')
print(f'\nCoeficientes (top 5 features):')
top_indices = np.argsort(np.abs(lr_model.coef_[0]))[-5:]
for idx in top_indices[::-1]:
    print(f'  {feature_names[idx]}: {lr_model.coef_[0][idx]:.4f}')

### O que observar

- A funcao sigmoid comprime qualquer valor real para [0, 1] -- perfeita para probabilidades
- BCE penaliza MUITO mais predicoes confiantes e erradas (predizer 0.1 quando eh 1 = loss alto)
- Os coeficientes da LR mostram direcao (positivo = aumenta P(classe 1)) e magnitude
- Acuracia treino vs teste indica se ha overfitting (se treino >> teste, ha overfitting)
- Top features por coeficiente revelam quais medidas mais indicam benignidade/malignidade

### O que concluir

Regressao Logistica eh um modelo LINEAR: captura apenas relacoes lineares entre features e classe.
Isso eh uma forca (interpretabilidade, regularizacao natural) e uma fraqueza (nao captura interacoes).
Em problemas onde features sao bem engineered, LR frequentemente supera modelos complexos.

### Conexao com outros notebooks

- Em 0.3 (Algebra Linear) voce viu multiplicacao matriz-vetor -- LR eh exatamente w*x + b
- Em 0.2 (Probabilidade) voce viu distribuicoes -- LR modela P(y|x) explicitamente
- Em 2.2 (EDA) voce criou features -- a qualidade delas impacta diretamente LR

## 3. Arvores de Decisao

### Analogia / Intuicao

Uma arvore de decisao eh como um jogo de 20 perguntas: a cada no, faz uma pergunta
("raio medio > 14.5?") e divide os dados em dois grupos. O objetivo eh que cada
grupo final (folha) seja o mais "puro" possivel (so uma classe).

### Definicao Formal

Arvores de decisao constroem splits recursivos minimizando impureza:
- **Gini:** 1 - sum(p_i^2) -- probabilidade de classificar errado uma amostra aleatoria
- **Entropia:** -sum(p_i * log(p_i)) -- quantidade de informacao necessaria

A cada no, escolhe a feature e o threshold que MAIS reduz a impureza.

### Por que em ML?

Arvores sao o modelo mais interpretavel de ML: voce pode literalmente ler as regras.
Em contextos regulados (medicina, financas), interpretabilidade eh obrigatoria.
Alem disso, arvores sao a base de Random Forest e Gradient Boosting (os melhores modelos para dados tabulares).

In [ ]:
# Decision Tree
dt_model = DecisionTreeClassifier(max_depth=5, random_state=42, criterion='gini')
dt_model.fit(X_train_scaled, y_train)

y_pred_dt = dt_model.predict(X_test_scaled)

print('=== ARVORE DE DECISAO ===')
print(f'Acuracia Teste: {accuracy_score(y_test, y_pred_dt):.4f}')
print('\nTop 5 features importantes:')

feature_importance_dt = pd.DataFrame({
    'feature': feature_names,
    'importance': dt_model.feature_importances_
}).sort_values('importance', ascending=False).head()

print(feature_importance_dt.to_string(index=False))

In [ ]:
# Visualizar arvore simplificada (max_depth=3 para visualizacao)

dt_viz = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_viz.fit(X_train_scaled, y_train)

plt.figure(figsize=(20, 10))
plot_tree(dt_viz, feature_names=feature_names, class_names=['Maligno', 'Benigno'],
          filled=True, rounded=True, fontsize=10)
plt.title('Arvore de Decisao (max_depth=3) - Simplificada para Visualizacao')
plt.tight_layout()
plt.show()

### O que observar

- Cada no mostra: feature usada, threshold, Gini, numero de amostras, e classe majoritaria
- Nos mais proximos da raiz usam as features mais discriminativas
- Profundidade (max_depth) controla complexidade: rasa = underfitting, funda = overfitting
- Feature importances indicam quais features participam mais dos splits

### O que concluir

Arvores sao interpretaveis e capturam nao-linearidades, mas tendem a overfitting.
A solucao eh limitar profundidade (max_depth), minimo de amostras por folha (min_samples_leaf),
ou usar ensembles (Random Forest, Gradient Boosting) que combinam muitas arvores.

### Conexao com outros notebooks

- Gini e Entropia usam conceitos de probabilidade (0.2)
- Feature importances complementam a analise de EDA (2.2)
- Em 3.3 voce vai estudar arvores e ensembles em profundidade

## 4. Random Forest

### Analogia / Intuicao

Uma arvore de decisao eh como perguntar a UM especialista. Random Forest eh como
perguntar a 100 especialistas independentes e votar na resposta mais popular.
Cada especialista ve dados ligeiramente diferentes (bagging) e features diferentes
(feature sampling), entao seus erros nao se correlacionam.

### Definicao Formal

Random Forest combina N arvores de decisao treinadas em:
1. **Bootstrap samples** (amostragem com reposicao do treino)
2. **Feature subsets** (cada split usa sqrt(p) features aleatorias)

A predicao final eh a votacao majoritaria. Isso reduz variance sem aumentar bias.

### Por que em ML?

Random Forest eh considerado o "melhor modelo default" para dados tabulares.
Funciona bem sem tuning extensivo, lida com features categoricas e numericas,
e fornece feature importances e OOB score (estimativa de erro sem validation set).

In [ ]:
# Random Forest
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1,
    oob_score=True,
)
rf_model.fit(X_train_scaled, y_train)

y_pred_rf = rf_model.predict(X_test_scaled)
y_pred_proba_rf = rf_model.predict_proba(X_test_scaled)[:, 1]

print('=== RANDOM FOREST ===')
print(f'Acuracia Treino: {accuracy_score(y_train, rf_model.predict(X_train_scaled)):.4f}')
print(f'Acuracia Validacao: {accuracy_score(y_val, rf_model.predict(X_val_scaled)):.4f}')
print(f'Acuracia Teste: {accuracy_score(y_test, y_pred_rf):.4f}')
print(f'\nOOB Score: {rf_model.oob_score_:.4f}')

# Feature importance
feature_importance_rf = pd.DataFrame({
    'feature': feature_names,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
top_n = 10
plt.barh(range(top_n), feature_importance_rf['importance'].head(top_n).values)
plt.yticks(range(top_n), feature_importance_rf['feature'].head(top_n).values)
plt.xlabel('Importancia')
plt.title(f'Top {top_n} Features - Random Forest')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

### O que observar

- OOB Score (Out-of-Bag) estima a acuracia usando amostras que cada arvore NAO viu no treino
- Feature importances medem reducao de impureza ponderada pelo numero de amostras
- Comparar acuracia treino vs teste: se muito proximas, RF generalizou bem
- n_estimators=100 arvores eh um bom default; mais arvores = mais estavel mas mais lento

### O que concluir

Random Forest sacrifica interpretabilidade (100 arvores sao opacas) por performance.
Eh robusto a outliers, nao requer normalizacao, e raramente overfita se n_estimators eh grande.
Para problemas tabulares sem restricao de interpretabilidade, RF eh quase sempre top-3.

### Conexao com outros notebooks

- O conceito de bias-variance (0.8) eh a base teorica de por que ensembles funcionam
- Feature importances complementam a EDA (2.2) -- features importantes para o modelo vs para a analise
- Em 3.3 voce vai comparar RF com Gradient Boosting (XGBoost, LightGBM)

### Por que Feature Importances diferem entre Random Forest e Regressao Logistica?

**Random Forest importance:**
- Baseada em reducao de impureza (Gini)
- Captura relacoes nao-lineares e interacoes
- Features que fazem boas separacoes em subsets sao priorizadas

**Logistic Regression coeficientes:**
- Baseada em impacto linear na log-odds
- Captura apenas relacoes lineares
- Features correlacionadas podem ter coeficientes cancelados (multicolinearidade)

Conclusao: modelos diferentes "enxergam" os dados de formas diferentes.
Por isso comparar importancias entre modelos eh uma tecnica poderosa de analise.

## 5. Support Vector Machines (SVM)

### Analogia / Intuicao

Imagine duas nuvens de pontos (classes). SVM encontra a "rua mais larga" entre elas
(hiperplano de margem maxima). Os pontos mais proximos da rua sao os "vetores de suporte"
-- sao eles que definem a fronteira. O kernel trick permite "dobrar" o espaco para
separar classes que nao sao linearmente separaveis.

### Definicao Formal

SVM maximiza a margem entre classes: max 2/||w|| sujeito a y_i(w*x_i + b) >= 1.
Com kernel RBF: K(x,y) = exp(-gamma * ||x-y||^2), que mapeia os dados para
um espaco de dimensao infinita onde se tornam linearmente separaveis.

### Por que em ML?

SVM eh particularmente forte em: alta dimensionalidade (muitas features, poucas amostras),
dados com margem clara entre classes, e problemas onde kernel tricks sao uteis.
Em NLP e bioinformatica, SVM foi dominante por mais de uma decada.

In [ ]:
# SVM com kernel RBF
svm_model = SVC(kernel='rbf', C=10, gamma='scale', probability=True, random_state=42)
svm_model.fit(X_train_scaled, y_train)

y_pred_svm = svm_model.predict(X_test_scaled)
y_pred_proba_svm = svm_model.predict_proba(X_test_scaled)[:, 1]

print('=== SUPPORT VECTOR MACHINE (RBF) ===')
print(f'Acuracia Teste: {accuracy_score(y_test, y_pred_svm):.4f}')
print(f'Numero de vetores de suporte: {len(svm_model.support_vectors_)}')
print(f'Porcentagem de support vectors: {len(svm_model.support_vectors_) / len(X_train_scaled) * 100:.1f}%')

### O que observar

- Numero de support vectors indica a complexidade do modelo: poucos = margem clara, muitos = classes se sobrepoe
- Porcentagem de support vectors alta (>50%) sugere que as classes sao dificeis de separar
- C controla regularizacao: C alto = margem estreita (mais overfit), C baixo = margem larga (mais underfit)
- gamma controla a influencia de cada ponto: gamma alto = cada ponto influencia pouco (overfit), gamma baixo = influencia ampla

### O que concluir

SVM com kernel RBF eh um classificador poderoso mas caro computacionalmente (O(n^2) a O(n^3)).
Para datasets grandes (>100k amostras), considere aproximacoes (Nystrom, SGD).
A normalizacao eh OBRIGATORIA para SVM -- sem ela, features com escala maior dominam.

### Conexao com outros notebooks

- Em 0.3 (Algebra Linear) voce viu normas e produtos internos -- SVM depende de ambos
- Em 3.4 voce vai estudar SVM e kernels em profundidade
- A necessidade de normalizacao conecta com feature scaling de 2.2

## 6. Naive Bayes

### Analogia / Intuicao

Naive Bayes eh como um detetive que analisa cada pista INDEPENDENTEMENTE.
"O suspeito estava na cena?" (feature 1). "Tem motivo?" (feature 2).
Cada resposta atualiza a probabilidade de culpa via Teorema de Bayes.
A suposicao "naive" eh que as pistas sao independentes (o que raramente eh verdade, mas funciona surpreendentemente bem).

### Definicao Formal

P(y|x) = P(x|y) * P(y) / P(x)
Com a suposicao naive: P(x|y) = prod(P(x_i|y))
GaussianNB assume que cada P(x_i|y) segue uma distribuicao Normal.

### Por que em ML?

Naive Bayes eh ultrarapido (treino em O(n*d)), funciona bem com poucas amostras,
e eh o baseline classico para classificacao de texto (spam detection, sentiment analysis).
Apesar da suposicao de independencia ser violada, o modelo eh surpreendentemente competitivo.

In [ ]:
# Naive Bayes
nb_model = GaussianNB()
nb_model.fit(X_train_scaled, y_train)

y_pred_nb = nb_model.predict(X_test_scaled)

print('=== NAIVE BAYES ===')
print(f'Acuracia Teste: {accuracy_score(y_test, y_pred_nb):.4f}')

### O que observar

- Naive Bayes nao precisa de normalizacao (trabalha com distribuicoes, nao distancias)
- Treinamento eh instantaneo mesmo em datasets grandes (apenas calcula medias e variancias)
- A acuracia pode ser menor que modelos mais complexos, mas o tradeoff speed/acuracia eh excelente

### O que concluir

Naive Bayes eh o modelo mais rapido de treinar e servir. Use como baseline inicial
e para problemas onde velocidade eh critica (real-time classification, text processing).
Se NB ja tem 90% de acuracia, talvez voce nao precise de um modelo mais complexo.

### Conexao com outros notebooks

- O Teorema de Bayes foi estudado em 0.2 (Probabilidade)
- A suposicao de normalidade das features conecta com distribuicoes (0.2)
- Em NLP (modulos futuros), Naive Bayes sera o primeiro modelo de classificacao de texto

## 7. Metricas de Classificacao

### Analogia / Intuicao

Acuracia eh como dizer "acertei 95% das provas". Mas se 95% das questoes sao faceis,
isso nao impressiona. Em classificacao desbalanceada, Accuracy eh enganosa:
um modelo que SEMPRE prediz "benigno" teria 63% de acuracia no Breast Cancer.
Precisao, Recall e F1 capturam nuances que Accuracy perde.

### Definicao Formal

- **Precisao = TP / (TP + FP)** -- dos que predisse positivo, quantos acertou?
- **Recall = TP / (TP + FN)** -- dos que ERAM positivo, quantos encontrou?
- **F1 = 2 * Precisao * Recall / (Precisao + Recall)** -- media harmonica
- **AUC-ROC** -- area sob a curva ROC, resume performance em todos os thresholds

### Por que em ML?

A escolha da metrica depende do custo do erro. Em cancer: FN (nao detectar tumor) eh
muito pior que FP (alarme falso) -> otimizar RECALL. Em spam: FP (email importante
marcado como spam) eh pior que FN (spam no inbox) -> otimizar PRECISAO.

In [ ]:
# Comparacao de metricas para todos os modelos
models = {
    'Logistic Regression': y_pred_test,
    'Decision Tree': y_pred_dt,
    'Random Forest': y_pred_rf,
    'SVM': y_pred_svm,
    'Naive Bayes': y_pred_nb
}

metrics_df = []
for model_name, y_pred in models.items():
    metrics_df.append({
        'Modelo': model_name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred)
    })

print('\n=== COMPARACAO DE METRICAS ===')
print(metrics_df.to_string(index=False))

plt.figure(figsize=(12, 5))
metrics_df.set_index('Modelo')[['Accuracy', 'Precision', 'Recall', 'F1-Score']].plot(kind='bar')
plt.ylabel('Score')
plt.title('Comparacao de Metricas entre Modelos')
plt.xticks(rotation=45, ha='right')
plt.ylim([0.8, 1.0])
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Matriz de Confusao
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for idx, (model_name, y_pred) in enumerate(models.items()):
    cm = confusion_matrix(y_test, y_pred)
    plt.imshow(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx], cbar=False)
    axes[idx].set_title(f'{model_name}')
    axes[idx].set_ylabel('Verdadeiro')
    axes[idx].set_xlabel('Predito')
    axes[idx].set_xticklabels(['Maligno', 'Benigno'])
    axes[idx].set_yticklabels(['Maligno', 'Benigno'])

axes[5].remove()
plt.tight_layout()
plt.show()

### O que observar

- Modelos com alta Accuracy podem ter baixo Recall (nao detectam a classe minoritaria)
- A matriz de confusao mostra exatamente ONDE cada modelo erra (FP vs FN)
- F1-Score equilibra Precisao e Recall -- use quando ambos importam igualmente
- Modelos diferentes podem ter trade-offs: LR tem mais FP, DT tem mais FN

### O que concluir

Nunca use APENAS Accuracy para avaliar classificacao. Sempre analise:
Precisao (confianca das predicoes positivas), Recall (cobertura dos positivos reais),
e a matriz de confusao (padrao de erros). A metrica ideal depende do contexto do problema.

### Conexao com outros notebooks

- Em 0.8 (Validacao) voce viu a importancia de avaliar corretamente
- Em 2.2 (EDA) a analise de desbalanceamento antecipa quais metricas usar
- Em modulos futuros, essas metricas guiam a selecao final do modelo

## 8. Curva ROC e Precision-Recall

### Analogia / Intuicao

Imagine um detector de metais no aeroporto. Threshold baixo: apita para tudo
(alto Recall, baixa Precisao). Threshold alto: so apita para metais grandes
(baixo Recall, alta Precisao). A curva ROC mostra TAREFA DO ALUNOS os trade-offs possiveis
variando o threshold de 0 a 1.

### Definicao Formal

- **Curva ROC:** plota TPR (Recall) vs FPR em todos os thresholds. AUC = area sob a curva.
- **Curva PR:** plota Precisao vs Recall. Mais informativa em dados desbalanceados.
- **AUC-ROC = 0.5** -> modelo aleatorio. **AUC-ROC = 1.0** -> modelo perfeito.

### Por que em ML?

Threshold padrao (0.5) raramente eh otimo. Em problemas medicos, juridicos ou financeiros,
o custo de FP e FN sao muito diferentes. A curva ROC permite escolher o threshold
que minimiza o custo total para o problema especifico.

In [ ]:
# Curvas ROC
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

models_proba = {
    'Logistic Regression': y_pred_proba_test,
    'Random Forest': y_pred_proba_rf,
    'SVM': y_pred_proba_svm
}

for model_name, y_pred_proba in models_proba.items():
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    auc = roc_auc_score(y_test, y_pred_proba)
    axes[0].plot(fpr, tpr, label=f'{model_name} (AUC={auc:.3f})', linewidth=2)

axes[0].plot([0, 1], [0, 1], 'k--', label='Random Classifier', alpha=0.5)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('Curva ROC')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Curva Precision-Recall
for model_name, y_pred_proba in models_proba.items():
    precision_vals, recall_vals, _ = precision_recall_curve(y_test, y_pred_proba)
    axes[1].plot(recall_vals, precision_vals, label=model_name, linewidth=2)

axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Curva Precision-Recall')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('AUC-ROC Scores:')
for model_name, y_pred_proba in models_proba.items():
    auc = roc_auc_score(y_test, y_pred_proba)
    print(f'  {model_name}: {auc:.4f}')

### O que observar

- Modelos com curva ROC mais proxima do canto superior-esquerdo sao melhores
- AUC-ROC > 0.9 eh excelente, 0.8-0.9 eh bom, < 0.7 eh preocupante
- A curva Precision-Recall eh mais informativa que ROC em dados muito desbalanceados
- Modelos podem ter AUC similar mas trade-offs Precisao/Recall muito diferentes

### O que concluir

AUC-ROC resume a capacidade discriminativa do modelo em um unico numero.
Mas nao use AUC como unica metrica: em problemas desbalanceados, AUC pode ser alto
mesmo com performance ruim na classe minoritaria. Sempre complemente com PR curve.

### Conexao com outros notebooks

- O conceito de threshold conecta com funcao sigmoid da Regressao Logistica (secao 2)
- Em 0.2 (Probabilidade) voce viu TPR e FPR como probabilidades condicionais
- Em modulos futuros, threshold tuning sera parte do pipeline de deployment

## 9. Cross-Validation

### Analogia / Intuicao

Avaliar um modelo em um unico test set eh como julgar um aluno por uma unica prova:
pode ter tido sorte ou azar. Cross-validation eh como fazer 5 provas diferentes
e calcular a media -- uma estimativa muito mais confiavel.

### Definicao Formal

Stratified K-Fold CV:
1. Divide os dados em K folds, mantendo a proporcao de classes em cada fold
2. Para cada fold: treina em K-1 folds, testa no fold restante
3. Resultado: K scores, cuja media eh a estimativa e desvio padrao indica estabilidade

### Por que em ML?

CV resolve dois problemas: (1) aproveita melhor dados escassos (todo dado eh usado para
treino E teste), (2) estima a variancia da performance (um desvio padrao alto indica
que o modelo eh instavel). Na industria, resultados sem CV nao sao confiaveis.

In [ ]:
# Stratified K-Fold (mantem proporcao de classes em cada fold)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models_cv = {
    'Logistic Regression': LogisticRegression(max_iter=10000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=10, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
    'SVM': SVC(kernel='rbf', C=10, gamma='scale'),
    'Naive Bayes': GaussianNB()
}

cv_results = []
for model_name, model in models_cv.items():
    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=skf, scoring='f1')
    cv_results.append({
        'Modelo': model_name,
        'F1 Medio': cv_scores.mean(),
        'Desvio Padrao': cv_scores.std(),
        'Scores por Fold': cv_scores
    })
    print(f'{model_name}: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})')

# Visualizar distribuicao de scores
plt.figure(figsize=(12, 5))
fold_data = [(row['Modelo'], row['Scores por Fold']) for row in cv_results]
models_names = [x[0] for x in fold_data]
scores_data = [x[1] for x in fold_data]

plt.boxplot(scores_data, labels=models_names)
plt.ylabel('F1-Score')
plt.title('Distribuicao de Scores Cross-Validation (5-Fold)')
plt.xticks(rotation=45, ha='right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### O que observar

- O desvio padrao dos scores indica estabilidade: < 0.02 eh estavel, > 0.05 eh preocupante
- Modelos com media alta mas desvio alto podem ser arriscados em producao
- Stratified KFold eh obrigatorio em dados desbalanceados (mantem proporcao de classes)
- O boxplot mostra a distribuicao dos scores -- outliers indicam folds problematicos

### O que concluir

Sempre use CV em vez de um unico train-test split para comparar modelos.
O modelo com maior media E menor desvio padrao eh o mais confiavel.
5-fold eh o padrao da industria; 10-fold para datasets pequenos.

### Conexao com outros notebooks

- Em 0.8 (Validacao) voce viu a teoria de CV -- aqui eh a pratica
- Em 2.2 (EDA) voce verificou desbalanceamento -- isso determina se precisa stratified CV
- Em Grid Search (proxima secao), CV eh usado internamente para avaliar cada combinacao

## 10. Hiperparametros e Grid Search

### Analogia / Intuicao

Hiperparametros sao como "knobs" de um equipamento de som: voce ajusta volume,
bass, treble ate encontrar o som ideal. Grid Search testa TODAS as combinacoes
de knobs sistematicamente e escolhe a melhor.

### Definicao Formal

GridSearchCV testa todas as combinacoes do grid de hiperparametros usando CV:
- Define grid: {param1: [v1, v2], param2: [v3, v4]} -> 4 combinacoes
- Para cada combinacao: roda K-Fold CV e calcula media do score
- Resultado: melhores parametros + melhor score (estimativa honesta via CV)

### Por que em ML?

Hiperparametros default raramente sao otimos. Grid Search automatiza a busca
e usa CV para evitar overfitting nos hiperparametros. Para problemas reais,
tuning pode melhorar a performance em 2-5% (significativo em producao).

In [ ]:
# Grid Search para Random Forest
param_grid_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 15],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}

rf_base = RandomForestClassifier(random_state=42, n_jobs=-1)
grid_search_rf = GridSearchCV(rf_base, param_grid_rf, cv=5, scoring='f1', n_jobs=-1, verbose=1)
grid_search_rf.fit(X_train_scaled, y_train)

print('=== GRID SEARCH - RANDOM FOREST ===')
print(f'Melhores parametros: {grid_search_rf.best_params_}')
print(f'Best F1-Score (CV): {grid_search_rf.best_score_:.4f}')

# Avaliar melhor modelo
best_rf = grid_search_rf.best_estimator_
y_pred_best_rf = best_rf.predict(X_test_scaled)
print(f'Acuracia Teste: {accuracy_score(y_test, y_pred_best_rf):.4f}')
print(f'F1-Score Teste: {f1_score(y_test, y_pred_best_rf):.4f}')

In [ ]:
# Grid Search para SVM
param_grid_svm = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.001, 0.01]
}

svm_base = SVC(kernel='rbf', probability=True, random_state=42)
grid_search_svm = GridSearchCV(svm_base, param_grid_svm, cv=5, scoring='f1', n_jobs=-1, verbose=1)
grid_search_svm.fit(X_train_scaled, y_train)

print('\n=== GRID SEARCH - SVM ===')
print(f'Melhores parametros: {grid_search_svm.best_params_}')
print(f'Best F1-Score (CV): {grid_search_svm.best_score_:.4f}')

best_svm = grid_search_svm.best_estimator_
y_pred_best_svm = best_svm.predict(X_test_scaled)
print(f'Acuracia Teste: {accuracy_score(y_test, y_pred_best_svm):.4f}')
print(f'F1-Score Teste: {f1_score(y_test, y_pred_best_svm):.4f}')

### O que observar

- Os melhores parametros encontrados podem diferir muito dos defaults
- O score do Grid Search (CV) pode ser menor que o score no test set -- isso eh normal
- Mais combinacoes no grid = busca mais exaustiva mas MUITO mais lenta
- n_jobs=-1 paraleliza a busca em todos os cores disponíveis

### O que concluir

Grid Search eh a forma mais simples de tuning, mas escala exponencialmente.
Para grids grandes, use RandomizedSearchCV (amostra combinacoes aleatorias) ou
Bayesian Optimization (Optuna, Hyperopt). O importante eh SEMPRE usar CV no tuning.

### Conexao com outros notebooks

- CV dentro do Grid Search conecta com secao 9 e com 0.8 (Validacao)
- Os hiperparametros tunados sao especificos de cada modelo (LR, RF, SVM)
- Em modulos futuros, tuning sera automatizado com Optuna

## 11. Exercicios Praticos

### Exercicio 1: Impacto da Normalizacao

Treine os 5 modelos COM e SEM normalizacao. Qual modelo eh mais sensivel?
Dica: modelos baseados em distancia (SVM, LR) sao mais afetados.

In [ ]:
# TAREFA DO ALUNO: Exercicio 1 - Comparar com/sem normalizacao
# Dica: treinar cada modelo com X_train e X_train_scaled, comparar acuracias
# Calcular a diferenca de acuracia para cada modelo

modelos = {
    'Logistic Regression': None,  # LogisticRegression(max_iter=10000, random_state=42)
    'Decision Tree': None,
    'Random Forest': None,
    'SVM': None,
    'Naive Bayes': None
}

# TAREFA DO ALUNO: para cada modelo, treinar com X_train e X_train_scaled
# TAREFA DO ALUNO: comparar acuracias no test set
# TAREFA DO ALUNO: qual modelo teve a MAIOR diferenca?

print('Comparacao com/sem normalizacao:')

In [ ]:
# SOLUCAO - Exercicio 1
from copy import deepcopy



modelos = {
    'Logistic Regression': LogisticRegression(max_iter=10000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=10, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(kernel='rbf', C=10, gamma='scale'),
    'Naive Bayes': GaussianNB()
}

print('=== IMPACTO DA NORMALIZACAO ===')
print(f'{"Modelo":<25} {"Sem Norm":>10} {"Com Norm":>10} {"Diferenca":>10}')
print('-' * 58)

for name, model in modelos.items():
    # Sem normalizacao
    m1 = deepcopy(model)
    m1.fit(X_train, y_train)
    acc_raw = accuracy_score(y_test, m1.predict(X_test))

    # Com normalizacao
    m2 = deepcopy(model)
    m2.fit(X_train_scaled, y_train)
    acc_scaled = accuracy_score(y_test, m2.predict(X_test_scaled))

    diff = abs(acc_raw - acc_scaled)
    print(f'{name:<25} {acc_raw:>10.4f} {acc_scaled:>10.4f} {diff:>10.4f}')

print()
print('Conclusao: SVM e LR sao mais sensiveis (baseados em distancia).')
print('Arvores e NB sao invariantes a escala.')

### Exercicio 2: Analise de Features

Compare as top 5 features por importancia no Random Forest com os top 5 coeficientes
da Regressao Logistica. As features sao as mesmas? Por que podem diferir?

In [ ]:
# TAREFA DO ALUNO: Exercicio 2 - Comparar feature importances RF vs LR
# Dica: use rf_model.feature_importances_ e lr_model.coef_[0]
# Ordene por magnitude e compare os top 5 de cada

top_rf = None  # DataFrame com top 5 features RF
top_lr = None  # DataFrame com top 5 features LR

print('Feature importances comparadas:')

In [ ]:
# SOLUCAO - Exercicio 2
import numpy as np

print('=== TOP 5 FEATURES ===')

# Random Forest
rf_imp = pd.DataFrame({
    'feature': feature_names,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)
print('\nRandom Forest (por reducao de Gini):')
print(rf_imp.head().to_string(index=False))

# Logistic Regression
lr_imp = pd.DataFrame({
    'feature': feature_names,
    'coef_abs': np.abs(lr_model.coef_[0])
}).sort_values('coef_abs', ascending=False)
print('\nLogistic Regression (por magnitude do coeficiente):')
print(lr_imp.head().to_string(index=False))

# Overlap
top5_rf = set(rf_imp.head()['feature'])
top5_lr = set(lr_imp.head()['feature'])
overlap = top5_rf & top5_lr
print(f'\nFeatures em comum no top 5: {len(overlap)} -> {overlap}')
print('RF captura nao-linearidades e interacoes; LR so ve impacto linear.')

### Exercicio 3: Threshold Otimo com Custo Customizado

Em diagnostico de cancer, nao detectar um tumor (FN) custa 10x mais que um alarme falso (FP).
Implemente uma funcao que encontre o threshold que minimiza o custo total.

In [ ]:
# TAREFA DO ALUNO: Exercicio 3 - Threshold otimo
# Dica: iterar thresholds de 0 a 1, calcular custo = cost_fn * FN + cost_fp * FP
# Usar confusion_matrix para obter FN e FP em cada threshold

def find_optimal_threshold(y_true, y_proba, cost_fn=10.0, cost_fp=1.0):
    # TAREFA DO ALUNO: implementar
    optimal_threshold = None
    return optimal_threshold

# threshold = find_optimal_threshold(y_test, y_pred_proba_test, cost_fn=10, cost_fp=1)
print('Threshold otimo encontrado!')

In [ ]:
# SOLUCAO - Exercicio 3
import numpy as np

def find_optimal_threshold(y_true, y_proba, cost_fn=10.0, cost_fp=1.0):
    thresholds = np.linspace(0.01, 0.99, 200)
    best_cost = float('inf')
    best_t = 0.5

    for t in thresholds:
        y_pred = (y_proba >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        cost = cost_fn * fn + cost_fp * fp
        if cost < best_cost:
            best_cost = cost
            best_t = t
    return best_t, best_cost

t_opt, cost_opt = find_optimal_threshold(y_test, y_pred_proba_test, cost_fn=10, cost_fp=1)

print(f'=== THRESHOLD OTIMO (FN custa 10x FP) ===')
print(f'Threshold otimo: {t_opt:.3f} (default: 0.500)')

# Comparar
for t, label in [(0.5, 'Default (0.5)'), (t_opt, f'Otimo ({t_opt:.3f})')]:
    y_p = (y_pred_proba_test >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_p).ravel()
    print(f'\n{label}:')
    print(f'  Recall: {recall_score(y_test, y_p):.4f}, Precisao: {precision_score(y_test, y_p):.4f}')
    print(f'  FN: {fn}, FP: {fp}, Custo total: {10*fn + 1*fp}')

### O que observar nos exercicios

- Normalizacao (Ex 1) impacta dramaticamente modelos baseados em distancia (SVM, LR) mas nao arvores
- Feature importances (Ex 2) diferem entre modelos lineares e nao-lineares -- ambas perspectivas sao validas
- Threshold tuning (Ex 3) pode reduzir custos significativamente sem retreinar o modelo

### O que concluir

Os exercicios mostram que classificacao nao eh so treinar um modelo: envolve
preprocessamento (normalizacao), analise (feature importances), e pos-processamento (threshold).
Cada etapa impacta o resultado final tanto quanto a escolha do algoritmo.

### Por que em ML?

Na industria, threshold tuning e feature analysis sao tao importantes quanto o modelo.
Um SVM com threshold otimizado pode superar um Random Forest com threshold default.

### Conexao com outros notebooks

- Normalizacao conecta com scaling de 2.2
- Feature analysis complementa EDA de 2.2
- Threshold tuning sera usado em deployment (modulos futuros)

### O que observar no panorama geral

- 5 modelos diferentes produzem resultados diferentes no MESMO dataset -- nao existe modelo universalmente melhor
- A pipeline completa (normalizar -> treinar -> avaliar -> tunar -> CV) eh mais importante que qualquer modelo
- Metricas devem ser escolhidas pelo CONTEXTO DO PROBLEMA, nao pela conveniencia

### O que concluir

Classificacao eh o ponto de partida de ML supervisionado. Os conceitos aprendidos aqui
(bias-variance, metricas, CV, tuning) se aplicam a TAREFA DO ALUNOS os modelos futuros.
O fluxo normalizar -> treinar -> avaliar -> tunar eh universal em ML.

### Por que em ML?

Esses fundamentos sao pre-requisitos para: deep learning, NLP, computer vision,
sistemas de recomendacao, e qualquer aplicacao avancada de ML.

### Conexao com outros notebooks

- Este notebook conecta fundamentos (Modulo 0) com pratica (Modulo 3)
- Os modelos estudados aqui serao detalhados em 3.2 (Regressao), 3.3 (Arvores), 3.4 (SVM)
- As metricas e CV serao usados em TAREFA DO ALUNOS os notebooks futuros

## 12. Erros Comuns e Armadilhas

### Erro 1: Data Leakage no Scaler

Fazer `scaler.fit(X)` em TAREFA DO ALUNOS os dados antes do split. Resultado: o scaler
"viu" o test set durante o treino, inflando as metricas.
Solucao: SEMPRE fit no train, transform no test. Pipeline do sklearn garante isso.

### Erro 2: Avaliar com Accuracy em dados desbalanceados

Dataset com 95% classe 0: modelo que SEMPRE prediz 0 tem 95% de accuracy.
Isso esconde performance terrivel na classe minoritaria.
Solucao: usar F1-Score, AUC-ROC, ou Precision/Recall.

### Erro 3: Nao estratificar o split

Em dados desbalanceados, um split aleatorio pode colocar 100% da classe minoritaria
no treino e 0% no teste (ou vice-versa).
Solucao: usar stratify=y no train_test_split e StratifiedKFold no CV.

### Erro 4: Threshold fixo em 0.5

O threshold default raramente eh otimo. Em problemas medicos (FN caro)
ou financeiros (FP caro), threshold deve ser ajustado ao custo.
Solucao: usar curva ROC ou funcao de custo para encontrar threshold otimo.

### Erro 5: Testar no set de treino

Acuracia no treino NAO indica performance real. Modelos complexos decoram o treino.
Solucao: SEMPRE avaliar em dados que o modelo NUNCA viu (test set ou CV).

### Erro 6: Grid Search sem CV

Tunar hiperparametros usando o test set causa overfitting nos hiperparametros.
O test set vira efetivamente um segundo treino.
Solucao: usar GridSearchCV com CV interno, avaliar no test set apenas UMA VEZ no final.

### Erro 7: Ignorar a baseline

Pular direto para modelos complexos sem estabelecer uma baseline simples (LR, NB).
Resultado: nao saber se o modelo complexo realmente vale a complexidade adicional.
Solucao: sempre comecar com LR ou NB como baseline.

## 13. Resumo e Conexoes

### Hierarquia de Conceitos

```
CLASSIFICACAO EM ML
|
+-- Modelos
|     |-- Regressao Logistica (linear, interpretavel, baseline)
|     |-- Arvore de Decisao (nao-linear, interpretavel, tende a overfit)
|     |-- Random Forest (ensemble, robusto, top performance)
|     |-- SVM (margem maxima, kernel trick, caro computacionalmente)
|     |-- Naive Bayes (probabilistico, ultrarapido, baseline para texto)
|
+-- Avaliacao
|     |-- Accuracy (simples, enganosa em desbalanceado)
|     |-- Precision / Recall / F1 (trade-off entre tipos de erro)
|     |-- AUC-ROC / PR Curve (performance em todos os thresholds)
|     |-- Matriz de Confusao (padrao de erros)
|
+-- Validacao
|     |-- Hold-out (train/val/test)
|     |-- Stratified K-Fold CV (estimativa robusta)
|     |-- Grid Search CV (tuning de hiperparametros)
|
+-- Otimizacao
      |-- Normalizacao (essencial para LR, SVM)
      |-- Threshold tuning (adaptar ao custo do problema)
      |-- Feature selection (via importances)
```

### Tabela de Conexoes

| Conceito | Onde apareceu antes | Onde sera usado |
|----------|-------------------|-----------------|
| Sigmoid / BCE | 0.2 (Probabilidade) | Deep Learning |
| Gini / Entropia | 0.2 (Probabilidade) | 3.3 (Arvores) |
| Bias-Variance | 0.8 (Validacao) | Todos os modulos |
| Feature Importance | 2.2 (EDA) | Feature Selection |
| Normalizacao | 2.2 (Scaling) | 3.4 (SVM), DL |
| Cross-Validation | 0.8 (Validacao) | Todos os modulos |

### Checklist de Competencias

- [ ] Sei treinar e avaliar Regressao Logistica, Arvore, RF, SVM e NB
- [ ] Entendo a diferenca entre Accuracy, Precision, Recall e F1
- [ ] Consigo interpretar curvas ROC e Precision-Recall
- [ ] Sei usar Stratified K-Fold Cross-Validation
- [ ] Consigo fazer Grid Search para tuning de hiperparametros
- [ ] Entendo quando e por que normalizar features
- [ ] Sei ajustar threshold baseado em custos do problema
- [ ] Consigo comparar feature importances entre modelos diferentes

### Proximos Passos

1. **3.2 Regressao:** Aplicar os mesmos conceitos (metricas, CV, tuning) para problemas de regressao
2. **3.3 Arvores e Ensemble:** Estudar em profundidade RF, Gradient Boosting, XGBoost
3. **3.4 SVM e Kernels:** Aprofundar em kernel trick, SVR, e aplicacoes
4. **3.5 Clustering:** Classificacao nao-supervisionada (sem labels)
5. **3.6 Reducao de Dimensionalidade:** PCA, t-SNE para visualizacao e feature reduction